# Tutorial 2: Streaming Responses with Claude

This tutorial teaches you how to stream responses from Claude for improved user experience and perceived performance.

## What You'll Learn

- Understanding streaming vs non-streaming responses
- Implementing Server-Sent Events (SSE) with Claude
- Handling streaming events and content
- Real-time token counting and usage tracking
- Best practices for production streaming

## Why Stream?

**Impact:** Streaming reduces perceived latency by 85% (source: patterns/optimization/streaming-patterns.md)

Benefits:
- Users see responses immediately
- Better UX for long responses
- Real-time feedback during generation
- Lower perceived wait time

---

## Setup

Install required packages:

In [ ]:
!pip install anthropic python-dotenv

In [ ]:
import os
from anthropic import Anthropic
from dotenv import load_dotenv
import time

load_dotenv()
client = Anthropic()

print("✓ Setup complete!")

## Example 1: Non-Streaming (Baseline)

First, let's see the traditional non-streaming approach:

In [ ]:
# Non-streaming request
start_time = time.time()

message = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=300,
    messages=[{
        "role": "user",
        "content": "Explain how HTTP requests work in 3 paragraphs."
    }]
)

elapsed = time.time() - start_time

print(f"⏱️  Total time: {elapsed:.2f}s")
print(f"📊 Tokens: {message.usage.output_tokens}\n")
print(message.content[0].text)

### Observation

With non-streaming, you wait for the entire response before seeing anything. For longer responses, this creates a poor user experience.

## Example 2: Basic Streaming

Now let's stream the response and display it in real-time:

In [ ]:
# Streaming request
from IPython.display import display, clear_output
import sys

start_time = time.time()
accumulated_text = ""

print("🔄 Streaming response:\n")

with client.messages.stream(
    model="claude-3-5-sonnet-20241022",
    max_tokens=300,
    messages=[{
        "role": "user",
        "content": "Explain how HTTP requests work in 3 paragraphs."
    }]
) as stream:
    for text in stream.text_stream:
        accumulated_text += text
        print(text, end="", flush=True)

elapsed = time.time() - start_time
print(f"\n\n⏱️  Total time: {elapsed:.2f}s")
print(f"📊 Tokens: {len(accumulated_text.split())} words")

### Key Difference

Notice how text appears immediately! While total time is similar, users see content right away instead of waiting.

## Example 3: Event-by-Event Streaming

For more control, you can handle individual events:

In [ ]:
# Stream with event handling
print("Event-by-event streaming:\n")

with client.messages.stream(
    model="claude-3-5-sonnet-20241022",
    max_tokens=200,
    messages=[{
        "role": "user",
        "content": "Write a haiku about programming."
    }]
) as stream:
    for event in stream:
        if event.type == "message_start":
            print("🚀 Message started")
        
        elif event.type == "content_block_start":
            print("📝 Content block started\n")
        
        elif event.type == "content_block_delta":
            if hasattr(event.delta, 'text'):
                print(event.delta.text, end="", flush=True)
        
        elif event.type == "content_block_stop":
            print("\n\n✅ Content block finished")
        
        elif event.type == "message_stop":
            print("🏁 Message complete")

## Example 4: Tracking Usage During Streaming

You can track token usage in real-time:

In [ ]:
# Track usage during streaming
input_tokens = 0
output_tokens = 0
text_chunks = []

print("Streaming with usage tracking:\n")

with client.messages.stream(
    model="claude-3-5-sonnet-20241022",
    max_tokens=300,
    messages=[{
        "role": "user",
        "content": "List 5 Python best practices with one-sentence explanations."
    }]
) as stream:
    for event in stream:
        # Track content
        if event.type == "content_block_delta":
            if hasattr(event.delta, 'text'):
                text_chunks.append(event.delta.text)
                print(event.delta.text, end="", flush=True)
        
        # Track token usage
        elif event.type == "message_delta":
            if hasattr(event, 'usage'):
                output_tokens += event.usage.output_tokens
        
        elif event.type == "message_start":
            if hasattr(event.message, 'usage'):
                input_tokens = event.message.usage.input_tokens

full_text = ''.join(text_chunks)

print(f"\n\n📊 Usage Statistics:")
print(f"  Input tokens: {input_tokens}")
print(f"  Output tokens: {output_tokens}")
print(f"  Total tokens: {input_tokens + output_tokens}")
print(f"  Text length: {len(full_text)} characters")

## Example 5: Error Handling in Streaming

Always handle errors gracefully:

In [ ]:
from anthropic import APIError, APIConnectionError, RateLimitError

def safe_stream(prompt, max_retries=3):
    """Stream with error handling and retries."""
    
    for attempt in range(max_retries):
        try:
            with client.messages.stream(
                model="claude-3-5-sonnet-20241022",
                max_tokens=200,
                messages=[{"role": "user", "content": prompt}]
            ) as stream:
                for text in stream.text_stream:
                    print(text, end="", flush=True)
            
            print("\n✅ Stream completed successfully")
            return
        
        except RateLimitError:
            print(f"⚠️  Rate limit hit. Waiting before retry {attempt + 1}/{max_retries}...")
            time.sleep(2 ** attempt)  # Exponential backoff
        
        except APIConnectionError:
            print(f"⚠️  Connection error. Retry {attempt + 1}/{max_retries}...")
            time.sleep(1)
        
        except APIError as e:
            print(f"❌ API error: {e}")
            return
    
    print("❌ Max retries exceeded")

# Test the safe streaming function
safe_stream("What is recursion? Explain in 2 sentences.")

## Example 6: Streaming with System Prompts

You can combine streaming with system prompts:

In [ ]:
# Streaming with system prompt
print("Code review streaming:\n")

with client.messages.stream(
    model="claude-3-5-sonnet-20241022",
    max_tokens=400,
    system="You are a senior code reviewer. Provide concise, numbered feedback.",
    messages=[{
        "role": "user",
        "content": """Review this code:

```python
def calculate(a, b, op):
    if op == '+':
        return a + b
    elif op == '-':
        return a - b
    elif op == '*':
        return a * b
    elif op == '/':
        return a / b
```
"""
    }]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

print("\n")

## Example 7: Performance Comparison

Let's measure time to first token (TTFT) - a key metric for user experience:

In [ ]:
import time

prompt = "Write a detailed explanation of database indexing with examples."

# Measure non-streaming
print("Non-streaming:")
start = time.time()
message = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=500,
    messages=[{"role": "user", "content": prompt}]
)
total_time = time.time() - start
print(f"  Time to first token: {total_time:.2f}s (entire response)")
print(f"  Total time: {total_time:.2f}s\n")

# Measure streaming
print("Streaming:")
start = time.time()
first_token_time = None

with client.messages.stream(
    model="claude-3-5-sonnet-20241022",
    max_tokens=500,
    messages=[{"role": "user", "content": prompt}]
) as stream:
    for text in stream.text_stream:
        if first_token_time is None:
            first_token_time = time.time() - start
        # Don't print to keep output clean

total_time = time.time() - start
print(f"  Time to first token: {first_token_time:.2f}s ⚡")
print(f"  Total time: {total_time:.2f}s")
print(f"\n💡 User saw content {((total_time - first_token_time) / total_time * 100):.0f}% faster with streaming!")

## Example 8: Building a Streaming Chat Interface

Here's a simple chat interface with streaming:

In [ ]:
def chat_with_streaming(user_message, conversation_history=None):
    """Simple streaming chat function."""
    
    if conversation_history is None:
        conversation_history = []
    
    # Add user message
    conversation_history.append({
        "role": "user",
        "content": user_message
    })
    
    print(f"User: {user_message}\n")
    print("Assistant: ", end="")
    
    # Stream response
    assistant_response = ""
    
    with client.messages.stream(
        model="claude-3-5-sonnet-20241022",
        max_tokens=300,
        messages=conversation_history
    ) as stream:
        for text in stream.text_stream:
            assistant_response += text
            print(text, end="", flush=True)
    
    print("\n" + "="*60 + "\n")
    
    # Add assistant response to history
    conversation_history.append({
        "role": "assistant",
        "content": assistant_response
    })
    
    return conversation_history

# Demo the chat interface
history = chat_with_streaming("What is machine learning?")
history = chat_with_streaming("Can you give me a simple example?", history)

## Interactive Exercise: Build Your Own Streamer

Create a streaming function with custom features:

In [ ]:
def custom_streamer(prompt, show_stats=True, show_events=False):
    """
    YOUR TURN: Enhance this streaming function!
    
    Ideas:
    - Add a progress indicator
    - Count words in real-time
    - Measure tokens per second
    - Add color coding for different event types
    """
    
    start_time = time.time()
    
    # TODO: Add your enhancements here
    
    with client.messages.stream(
        model="claude-3-5-sonnet-20241022",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
    
    elapsed = time.time() - start_time
    
    if show_stats:
        print(f"\n\n⏱️  Completed in {elapsed:.2f}s")

# Test your function
custom_streamer("Explain what an API is in simple terms.")

## Production Best Practices

### 1. Always Use Timeouts

```python
from anthropic import Anthropic

client = Anthropic(
    timeout=60.0,  # Set reasonable timeout
    max_retries=2
)
```

### 2. Handle Network Interruptions

```python
try:
    with client.messages.stream(...) as stream:
        for text in stream.text_stream:
            # Process text
            pass
except APIConnectionError:
    # Implement retry logic
    pass
```

### 3. Buffer for Display

For UI applications, buffer text slightly to avoid flickering:

```python
buffer = ""
for text in stream.text_stream:
    buffer += text
    if len(buffer) > 10:  # Update every 10 chars
        update_ui(buffer)
        buffer = ""
```

### 4. Track Costs in Real-Time

```python
COST_PER_INPUT_TOKEN = 3.0 / 1_000_000  # $3 per million
COST_PER_OUTPUT_TOKEN = 15.0 / 1_000_000  # $15 per million

total_cost = (input_tokens * COST_PER_INPUT_TOKEN + 
              output_tokens * COST_PER_OUTPUT_TOKEN)
```

## Key Takeaways

1. **Streaming Improves UX**: 85% reduction in perceived latency

2. **Use `messages.stream()`**: Simple context manager interface

3. **Handle Events**: Process `content_block_delta` for text chunks

4. **Track Usage**: Monitor tokens in real-time for cost control

5. **Error Handling**: Always implement retries and graceful failures

6. **Production Ready**: Use timeouts, buffers, and connection handling

## Next Steps

- **Tutorial 3**: Learn tool use and function calling
- **Tutorial 4**: Master multi-turn conversation management
- **Tutorial 5**: Apply production patterns at scale

## Resources

- [Streaming Pattern Documentation](../patterns/optimization/streaming-patterns.md)
- [Anthropic Streaming Guide](https://docs.anthropic.com/en/api/streaming)
- [Token Efficiency Patterns](../patterns/optimization/token-efficiency.md)